# Phase 2 — Fine-tune LayoutLMv3 on CORD (Colab GPU)

Runs on Colab with a GPU runtime. Trains, tracks to a local MLflow file-store on the Colab VM, and saves a bundle to download for `docintel-import-kie`.

## 1. Install

Push this branch first (`git push -u origin worktree-phase-2-kie`), or change `@worktree-phase-2-kie` to `@master` after the Phase 2 merge.

In [ ]:
# Use a GPU runtime: Runtime > Change runtime type > GPU.
!pip install -q "docintel[train,kie] @ git+https://github.com/KhoiDang1209/AI-Document-Understanding.git@worktree-phase-2-kie#subdirectory=docintel"

## 2. Setup

In [ ]:
import json
import subprocess
from pathlib import Path

import mlflow
from datasets import load_dataset
from transformers import LayoutLMv3Processor

from docintel.config import get_settings
from docintel.kie.config import TrainingConfig
from docintel.kie.dataset import collect_categories, encode_example, parse_cord_example
from docintel.kie.labels import build_label_list, build_label_maps
from docintel.kie.train import run_training

settings = get_settings()
config = TrainingConfig.from_settings(settings)
mlflow.set_tracking_uri("file:./mlruns")   # local file-store on the Colab VM
mlflow.set_experiment("cord-kie")

## 3. Data + labels

In [ ]:
DATASET_REVISION = "main"  # pin to a specific revision for reproducibility
raw = load_dataset("naver-clova-ix/cord-v2", revision=DATASET_REVISION)
train_gt = [json.loads(ex["ground_truth"]) for ex in raw["train"]]
label_list = build_label_list(collect_categories(train_gt))
id2label, label2id = build_label_maps(label_list)
processor = LayoutLMv3Processor.from_pretrained(config.model_name, apply_ocr=False)

## 4. Features

`encode_example` takes the image and returns the token encoding plus `pixel_values` (shape `(1, 3, 224, 224)`), so we squeeze the leading batch dim.

In [ ]:
def _encode_split(split):
    def _map(ex):
        gt = json.loads(ex["ground_truth"])
        words, boxes, bio = parse_cord_example(gt)
        enc = encode_example(ex["image"], words, boxes, bio, processor, label2id)
        enc["pixel_values"] = enc["pixel_values"][0]  # squeeze (1,3,224,224) -> (3,224,224)
        return enc
    return split.map(_map, remove_columns=split.column_names)

train_ds = _encode_split(raw["train"])
eval_ds = _encode_split(raw["validation"])

## 5. Train & track

In [ ]:
git_sha = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True
).stdout.strip() or "unknown"
bundle = run_training(
    config=config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    id2label=id2label,
    label2id=label2id,
    processor=processor,
    bundle_dir=Path("cord-layoutlmv3-bundle"),
    dataset_revision=DATASET_REVISION,
    git_sha=git_sha,
)
print("Bundle saved to:", bundle)

## 6. Download the bundle

In [ ]:
import shutil

from google.colab import files  # type: ignore

archive = shutil.make_archive("cord-layoutlmv3-bundle", "zip", "cord-layoutlmv3-bundle")
files.download(archive)

## 7. Next: import on the laptop

Download the zip, unzip it, then on the laptop (with docker-compose MLflow + MinIO up) run `docintel-import-kie --bundle-dir ./cord-layoutlmv3-bundle`. Then smoke-test on CPU: `from transformers import AutoModelForTokenClassification; AutoModelForTokenClassification.from_pretrained("cord-layoutlmv3-bundle/model")` loads and a single forward pass returns logits of shape `[1, seq, num_labels]`.